# `process(StateChange)` — Yeatman2021 ROAR (full pipeline)

Production-pipeline counterpart to [`state_change.ipynb`](state_change.ipynb). Same induced-dyslexia experimental shape, but:

- Uses the actual `Yeatman2021-lexical_decision-image` benchmark (200 train + 100 test ROAR stimuli)
- Scores via `benchmark(bs_model)` → routes through the full `bs_model.process(stimulus_set)` pipeline including the registration's `generation_fn`
- Returns published-comparable numbers (Qwen baseline ≈ 0.93 on ROAR, paper threshold = 0.65 for dyslexia)

The four brain-score calls are unchanged:

```python
bs_model = brainscore.load_model('qwen2.5-vl-3b')
bs_model._state_change_fn = build_pytorch_ablation_fn(bs_model._model)
applied = bs_model.process(StateChange(...))
bs_model.reset()
```

Plus one call to score: `score = benchmark(bs_model)`.

**Hardware note**: ~40-55 min on Mac MPS, ~15-20 min on EC2 g5.4xlarge. Stimulus images stream from the brain-score S3 bucket on first run (~50 MB).

In [ ]:
import random
import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image

import brainscore
import brainscore_vision  # registers Qwen2.5-VL
from brainscore_core.model_interface import StateChange, Selection, Perturbation
from brainscore.perturbation import build_pytorch_ablation_fn


def resolve(model, path):
    """Walk a dotted path on a torch module."""
    for p in path.split('.'):
        model = model[int(p)] if p.isdigit() else getattr(model, p)
    return model


## 1 · Load the benchmark and model

`load_benchmark` materializes the Yeatman2021 stimulus set (downloads from S3 on first run). `load_model` constructs a registered `BrainScoreModel`. The `state_change_fn` wiring is identical to the toy notebook — explicit by design, so brainscore_core stays torch-free.

In [ ]:
benchmark = brainscore.load_benchmark('Yeatman2021-lexical_decision-image')
bs_model = brainscore.load_model('qwen2.5-vl-3b')

# Wiring is explicit on purpose: brainscore_core stays torch-free, and
# non-pytorch models or custom ablation strategies pass their own
# callable here.
bs_model._state_change_fn = build_pytorch_ablation_fn(bs_model._model)

device = ('mps' if torch.backends.mps.is_available()
          else 'cuda' if torch.cuda.is_available() else 'cpu')
qwen = bs_model._model.to(device).eval()
processor = bs_model._preprocessors['vision']._processor

print(f'benchmark:    {benchmark.identifier}')
print(f'train stims:  {len(benchmark._train_stimuli)} '
      f'(real + pseudo, used as localizer)')
print(f'test stims:   {len(benchmark._test_stimuli)} (held-out)')
print(f'human ceiling: {float(benchmark.ceiling):.3f}')
print(f'device:        {device}')


## 2 · Inspect the stimuli

ROAR stimuli are 500×300 black-text-on-white PNGs (the original Yeatman et al. 2021 paradigm — same images human dyslexia screening uses).

In [ ]:
train_stim = benchmark._train_stimuli
real_paths   = train_stim[train_stim['image_label'] == 'real']['image_file_name'].tolist()
pseudo_paths = train_stim[train_stim['image_label'] == 'pseudo']['image_file_name'].tolist()

fig, axes = plt.subplots(2, 4, figsize=(11, 4))
for ax, p in zip(axes[0], real_paths[:4]):
    ax.imshow(Image.open(p)); ax.set_xticks([]); ax.set_yticks([])
for ax, p in zip(axes[1], pseudo_paths[:4]):
    ax.imshow(Image.open(p)); ax.set_xticks([]); ax.set_yticks([])
# Row labels signal real-vs-pseudo; the image content itself IS the
# stimulus word, so per-cell titles would just duplicate the image.
axes[0, 0].set_ylabel(f'real\n(n={len(real_paths)})',
                     rotation=0, ha='right', va='center', fontsize=11)
axes[1, 0].set_ylabel(f'pseudo\n(n={len(pseudo_paths)})',
                     rotation=0, ha='right', va='center', fontsize=11)
plt.suptitle('ROAR stimulus examples (4 train images per condition)', fontsize=11)
plt.tight_layout(); plt.show()


## 3 · Localize word-selective units across late MLPs

Subsample 50 real + 50 pseudo from the 200+200 train split (full set would be ~25 min just for localization on Mac MPS; 50+50 captures the contrast and runs in ~5 min).

Multi-hook localization: register hooks on all 5 layers simultaneously, do one forward pass per stimulus, capture all 5 layers' activations at once. 5× faster than localizing each layer separately.

In [ ]:
PROMPT_LOC = 'What word is shown in this image?'

def _generate_localize(image, max_new_tokens=1):
    """One-token forward pass to fire the localizer hooks."""
    msg = [{'role': 'user', 'content': [
        {'type': 'image', 'image': image},
        {'type': 'text', 'text': PROMPT_LOC}]}]
    text = processor.apply_chat_template(msg, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[image], return_tensors='pt').to(device)
    with torch.no_grad():
        qwen.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)

def localize_multi(real_imgs, pseudo_imgs, layer_paths, k):
    """Per-layer top-k word-selective units. Cohen's d-style selectivity."""
    captured = {lp: [] for lp in layer_paths}
    handles = []
    for lp in layer_paths:
        layer = resolve(qwen, lp)
        def hook(_m, _i, output, _lp=lp):
            h = output[0] if isinstance(output, tuple) else output
            captured[_lp].append(h.detach().to('cpu', dtype=torch.float32)
                                  .mean(dim=1).squeeze(0).numpy())
        handles.append(layer.register_forward_hook(hook))
    try:
        for img in real_imgs + pseudo_imgs:
            _generate_localize(img)
    finally:
        for h in handles: h.remove()
    per_layer_units, selectivity_by_layer = {}, {}
    for lp in layer_paths:
        acts = np.stack(captured[lp])
        real_act, pseudo_act = acts[:len(real_imgs)], acts[len(real_imgs):]
        pooled = np.sqrt((real_act.var(0) + pseudo_act.var(0)) / 2 + 1e-6)
        sel = (real_act.mean(0) - pseudo_act.mean(0)) / pooled
        per_layer_units[lp] = np.argsort(sel)[-k:].tolist()
        selectivity_by_layer[lp] = sel
    return per_layer_units, selectivity_by_layer

random.seed(0)
LOC_N      = 50
loc_real   = [Image.open(p).convert('RGB') for p in random.sample(real_paths,   LOC_N)]
loc_pseudo = [Image.open(p).convert('RGB') for p in random.sample(pseudo_paths, LOC_N)]

LAYERS = [f'model.language_model.layers.{i}.mlp' for i in (26,28,30,32,34)]
TOP_K  = 1500
per_layer_units, selectivity_by_layer = localize_multi(
    loc_real, loc_pseudo, LAYERS, TOP_K)

for lp in LAYERS:
    s = selectivity_by_layer[lp]
    print(f'  {lp:42s}  selectivity range [{s.min():+.2f}, {s.max():+.2f}]')


## 4 · Run the experiment via the full benchmark pipeline

`benchmark(bs_model)` is the production scoring path. Internally it does:
1. `bs_model.start_task(TaskContext(instruction=..., label_set=['real','pseudo'], task_type='probabilities'))`
2. `predictions = bs_model.process(test_stimuli)` — dispatches to `bs_model._generation_fn` (Qwen's chat-template + parse closure) for each test image
3. Computes accuracy, normalizes by ceiling, returns `Score`

The `state_change` hooks installed below are active during step 2's forward passes — that's how the lesion affects the benchmark score.

In [ ]:
# (a) baseline — no perturbation
baseline = benchmark(bs_model)
print(f'baseline:  raw={float(baseline.attrs["raw"]):.3f}  '
      f'real-acc={baseline.attrs["accuracy_real"]:.3f}  '
      f'pseudo-acc={baseline.attrs["accuracy_pseudo"]:.3f}')

# (b) install the lesion at every selected layer. Each process(StateChange)
# call dispatches to bs_model._state_change_fn, installs a forward hook,
# and registers a cleanup handle in bs_model._active_perturbations.
for layer, units in per_layer_units.items():
    bs_model.process(StateChange(
        kind='ablation',
        target=Selection(layer=layer, indices=units),
        perturbation=Perturbation(kind='zero'),
    ))

# (c) score under lesion — hooks fire on every test-stimulus forward pass
lesioned = benchmark(bs_model)
print(f'lesioned:  raw={float(lesioned.attrs["raw"]):.3f}  '
      f'real-acc={lesioned.attrs["accuracy_real"]:.3f}  '
      f'pseudo-acc={lesioned.attrs["accuracy_pseudo"]:.3f}')

# (d) reset() invokes every cleanup in _active_perturbations and clears
# the registry. To remove a single perturbation without touching the
# others, use process(StateChange(kind='reset', handle_id=...)).
bs_model.reset()
restored = benchmark(bs_model)
print(f'restored:  raw={float(restored.attrs["raw"]):.3f}  '
      f'real-acc={restored.attrs["accuracy_real"]:.3f}  '
      f'pseudo-acc={restored.attrs["accuracy_pseudo"]:.3f}')

DYSLEXIA_THRESHOLD = 0.65
print()
print(f'lesioned dyslexic? {lesioned.attrs["dyslexic"]}  '
      f'(threshold = {DYSLEXIA_THRESHOLD})')


## 5 · Visualize

In [ ]:
fig = plt.figure(figsize=(13, 4.2))
gs = fig.add_gridspec(1, 2, width_ratios=[2, 3], wspace=0.3)
ax_a = fig.add_subplot(gs[0]); ax_b = fig.add_subplot(gs[1])

# Panel A — selectivity at the last lesioned MLP
last = LAYERS[-1]
sel  = selectivity_by_layer[last]
thr  = sel[per_layer_units[last]].min()
ax_a.hist(sel, bins=80, color='#4a6fa5', edgecolor='white', linewidth=0.4)
ax_a.axvline(thr, color='black', lw=1, ls='--')
ax_a.text(thr, ax_a.get_ylim()[1] * 0.92, f'  top {TOP_K}', ha='left', fontsize=9)
ax_a.set_xlabel(r'$\bar{a}_\mathrm{real} - \bar{a}_\mathrm{pseudo}$ (Cohen$\,d$)')
ax_a.set_ylabel('Units')
ax_a.set_title('A. Word selectivity (last lesioned MLP)', loc='left', fontsize=11)

# Panel B — split-by-class accuracy across conditions, with dyslexia threshold
conds  = ['baseline', 'lesioned', 'restored']
scores = [baseline, lesioned, restored]
raw    = [float(s.attrs['raw'])             for s in scores]
real_a = [s.attrs['accuracy_real']          for s in scores]
pseudo_a = [s.attrs['accuracy_pseudo']      for s in scores]

x = np.arange(len(conds)); w = 0.27
ax_b.bar(x - w, raw,      width=w, color='#4a6fa5', label='overall')
ax_b.bar(x,     real_a,   width=w, color='#3aa55a', label='real-word acc')
ax_b.bar(x + w, pseudo_a, width=w, color='#a54a4a', label='pseudo-word acc')
ax_b.axhline(DYSLEXIA_THRESHOLD, color='black', lw=0.6, ls=':')
ax_b.text(len(conds) - 0.5, DYSLEXIA_THRESHOLD + 0.02,
          'dyslexia threshold (0.65)', ha='right', fontsize=9, color='black')
ax_b.set_xticks(x); ax_b.set_xticklabels(conds, fontsize=10)
ax_b.set_ylabel('Accuracy'); ax_b.set_ylim(0, 1.05)
ax_b.set_title(f'B. Yeatman2021 lexical decision  '
               f'(n_test={float(baseline.attrs["n_test_stimuli"]):.0f})',
               loc='left', fontsize=11)
ax_b.legend(loc='lower right', fontsize=9)
for i, r in enumerate(raw):
    ax_b.text(i - w, r + 0.02, f'{r:.2f}', ha='center', fontsize=9)

for ax in (ax_a, ax_b):
    for sp in ('top', 'right'): ax.spines[sp].set_visible(False)
plt.show()


## Comparison to published numbers

From Brain-Score's prior scoring runs (CLAUDE.md milestone log):

| Model | Baseline ROAR (no lesion) | Path |
|---|---|---|
| Qwen2.5-VL-3B | **0.930** | generation |
| BLIP-2 OPT-2.7B | 0.500 | generation (chance) |
| GPT-2 (text only) | 0.810 | readout |
| CLIP ViT-B/32 | 0.680 | readout |
| chance baseline | 0.500 | — |

Our `baseline` here should reproduce ~0.93 (within sampling noise — the benchmark is deterministic given the seed, but TaskContext + chat-template details can shift slightly across transformers versions).

**The induced-dyslexia signature** (Honarmand et al. 2026, ICLR): the `lesioned` score should drop *substantially*, with the drop concentrated in `accuracy_real` (the dyslexia pattern — pseudo-words are unaffected because saying 'pseudo' is the default failure mode). Crossing the 0.65 threshold makes the model 'dyslexic' by the paper's definition.

**Caveats** vs the paper's actual replication:
- The paper uses Qwen-72B (80 transformer blocks); we use 3B (36). Smaller model, more concentrated lesion required.
- The paper's localizer has 4 control conditions (real, scrambled, line objects, line faces); ours has 2 (real, pseudo). Coarser selectivity.
- The paper distributes ablation across all 80 blocks; we hit 5 late blocks. Same idea, fewer points.

Despite the simplifications the *experimental shape* — localize → ablate → behavioral test → reset, all from a single `BrainScoreModel` registration via `process(StateChange)` — matches.